In [3]:
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[1]))
import utils.tools as tools

In [4]:
# Suma kontrolna
nazwa_pliku = "kalendarz_pelny_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"../../dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   kalendarz_pelny_towid.parquet   defac931291e6bad894c972b985ab043503fa4bb1ee673152bdcc24d27aa7c17


# Flagi towarowe
**CzyWazone — towar sprzedawany na wagę**

In [5]:
df_kalendarz = pd.read_parquet("dane/interim/kalendarz_pelny_towid.parquet")

FileNotFoundError: [Errno 2] No such file or directory: 'dane/interim/kalendarz_pelny_towid.parquet'

### Towary wazone 

In [ ]:
# ============================================================
# SOFT FILTER — Kryterium 2: Towary ważone (per TowId)
# Krok 1: % transakcji z ułamkową ilością per TowId
# ============================================================

wazone_check = df_kalendarz.groupby('TowId').agg(
    LiczbaTransakcji=('IloscPlus', 'count'),
    LiczbaUlamkowych=('IloscPlus', lambda x: (x % 1 != 0).sum()),
).reset_index()

wazone_check['PctUlamkowych'] = (
    wazone_check['LiczbaUlamkowych'] / wazone_check['LiczbaTransakcji'] * 100
)

print(wazone_check['PctUlamkowych'].describe())

count    12481.000000
mean         0.680205
std          6.620668
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         98.976843
Name: PctUlamkowych, dtype: float64


In [ ]:


def czy_nazwa_sugeruje_wage(nazwa):
    nazwa = str(nazwa).upper().strip()
    
    # Wyklucz przypadki "liczba+KG" (stała gramatura opakowania, np. "0,5KG", "1,65KG")
    bez_gramatury = re.sub(r'\d+[\.,]?\d*\s*KG\b', '', nazwa)
    
    # Szukamy samodzielnego "KG" jako jednostki sprzedaży (bez liczby bezpośrednio przed nim)
    return bool(re.search(r'\bKG\b', bez_gramatury))

# Szybki test kontrolny
testy = [
    "Chleb mieszany 0.6 kg PRECELEK",         # False - gramatura
    "PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL", # False - gramatura
    "MĄKA ZIEMNIACZANA 0,5KG",                # False - gramatura
    "SURÓWKA KG",                              # True - waga
    "BAKŁAŻAN KG POLSKA",                      # True - waga
    "SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK",      # True - waga
]
for t in testy:
    print(f"{czy_nazwa_sugeruje_wage(t)}: {t}")

False: Chleb mieszany 0.6 kg PRECELEK
False: PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL
False: MĄKA ZIEMNIACZANA 0,5KG
True: SURÓWKA KG
True: BAKŁAŻAN KG POLSKA
True: SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK


In [ ]:
nazwy_do_sprawdzenia = df_kalendarz[['TowId', 'NazwaTow']].drop_duplicates(subset='TowId')
nazwy_do_sprawdzenia['NazwaSugerujeWage'] = nazwy_do_sprawdzenia['NazwaTow'].apply(czy_nazwa_sugeruje_wage)

wazone_check_pelne = wazone_check.merge(nazwy_do_sprawdzenia, on='TowId', how='left')

wazone_check_pelne['JestWazony'] = (
    (wazone_check_pelne['PctUlamkowych'] >= 50) |
    (wazone_check_pelne['NazwaSugerujeWage'])
)

lista_wazonych_towid = set(wazone_check_pelne[wazone_check_pelne['JestWazony']]['TowId'])
print(f"Ważonych TowId (finalne kryterium): {len(lista_wazonych_towid)}")

dodane_przez_nazwe = wazone_check_pelne[
    (wazone_check_pelne['NazwaSugerujeWage']) & 
    (wazone_check_pelne['PctUlamkowych'] < 50)
]
print(f"\nDodatkowo złapane przez samodzielne 'KG' (bez liczby): {len(dodane_przez_nazwe)}")
print(dodane_przez_nazwe[['TowId', 'NazwaTow', 'PctUlamkowych']].sort_values('NazwaTow'))

Ważonych TowId (finalne kryterium): 195

Dodatkowo złapane przez samodzielne 'KG' (bez liczby): 118
       TowId                           NazwaTow  PctUlamkowych
4988   51262                          ANANAS KG       9.090909
5390   52141                 BAKŁAŻAN KG POLSKA       6.666667
3564   25652      BRZUSZKI Z ŁOSOSIA WĘDZONE KG       7.425743
4382   49625                          BURAKI KG       9.330820
9166   77611      CIASTKA MARKIZY KG  DR GERARD       8.152174
...      ...                                ...            ...
2735   16441              SER SALAMI MLEKPOL KG      43.102163
2748   16479   SER Z ORZECHEM WŁOSKIM KG ŁOWICZ      45.038168
11332  80088                         SURÓWKA KG       8.000000
5853   53068  USZKA Z GRZYBAMI KG KUCHNIA POLKI       1.683938
8870   67234               ZIEMNIAKI SŁODKIE KG      35.164835

[118 rows x 3 columns]


In [ ]:
df_kalendarz['JestWazony'] = df_kalendarz['TowId'].isin(lista_wazonych_towid)

print(df_kalendarz['JestWazony'].value_counts())

nazwy_wazonych = (
    df_kalendarz[df_kalendarz['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    8463825
True      338690
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    42
SERY WAGA /nabiał                33
WARZYWA                          31
OWOCE                            19
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
PIECZYWO                          3
WARZYWA KISZONE                   3
RYBY WĘDZONE WAGA /ryby           3
RYBY MROŻONE WAGA                 2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
WIELKANOCNE                       1
***PRZECENY                       1
CHLEBY                            1
ŚWIĄTECZNE                        1
CIASTKA                           1
MIĘSO, WĘDLINY I GARMAŻERKA       1
Name: count, dtype: int64


In [ ]:
df_kalendarz['JestWazony'] = df_kalendarz['TowId'].isin(lista_wazonych_towid)

print(df_kalendarz['JestWazony'].value_counts())

nazwy_wazonych = (
    df_kalendarz[df_kalendarz['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    8463825
True      338690
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    42
SERY WAGA /nabiał                33
WARZYWA                          31
OWOCE                            19
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
PIECZYWO                          3
WARZYWA KISZONE                   3
RYBY WĘDZONE WAGA /ryby           3
RYBY MROŻONE WAGA                 2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
WIELKANOCNE                       1
***PRZECENY                       1
CHLEBY                            1
ŚWIĄTECZNE                        1
CIASTKA                           1
MIĘSO, WĘDLINY I GARMAŻERKA       1
Name: count, dtype: int64


## CzyMartwy — brak sprzedaży w ostatnich N dniach 

In [ ]:
# ============================================================
# CzyMartwy — z uwzględnieniem sezonowości (próg per-TowId)
# ============================================================

df_kalendarz = df_kalendarz.sort_values(['TowId', 'Data']).reset_index(drop=True)

# 1) Data ostatniej faktycznej sprzedaży (forward-fill w obrębie TowId)
df_kalendarz['DataOstatniejSprzedazy'] = df_kalendarz['Data'].where(df_kalendarz['DokId'] != -1)
df_kalendarz['DataOstatniejSprzedazy'] = df_kalendarz.groupby('TowId')['DataOstatniejSprzedazy'].ffill()

df_kalendarz['DniOdSprzedazy'] = (
    df_kalendarz['Data'] - df_kalendarz['DataOstatniejSprzedazy']
).dt.days

# 2) Historyczne przerwy między KOLEJNYMI dniami sprzedaży, per TowId
dni_sprzedazy = (
    df_kalendarz[df_kalendarz['DokId'] != -1]
    [['TowId', 'Data']]
    .drop_duplicates()
    .sort_values(['TowId', 'Data'])
)
dni_sprzedazy['PoprzedniaData'] = dni_sprzedazy.groupby('TowId')['Data'].shift(1)
dni_sprzedazy['Przerwa'] = (dni_sprzedazy['Data'] - dni_sprzedazy['PoprzedniaData']).dt.days

# 3) Dla każdego TowId: jego własny "rekord" najdłuższej normalnej przerwy (np. 95 percentyl -
#    odporne na pojedynczy wyjątkowy przypadek, w przeciwieństwie do max())
prog_indywidualny = (
    dni_sprzedazy.groupby('TowId')['Przerwa']
    .quantile(0.95)
    .rename('PrzerwaHistoryczna95pct')
)

# 4) Próg per TowId = margines nad własną historią, z podłogą (minimum) dla produktów
#    z krótką historią, gdzie 95 percentyl nic jeszcze nie mówi
N_MINIMUM = 550          # dotychczasowy próg — podłoga dla produktów bez wyraźnej sezonowości
MARGINES = 1.5           # ile razy dłuższa niż zwykle musi być przerwa, żeby uznać za "martwy"

df_kalendarz = df_kalendarz.merge(prog_indywidualny, on='TowId', how='left')
df_kalendarz['PrognMartwy'] = np.maximum(
    N_MINIMUM,
    df_kalendarz['PrzerwaHistoryczna95pct'] * MARGINES
)

df_kalendarz['CzyMartwyWTymDniu'] = df_kalendarz['DniOdSprzedazy'] >= df_kalendarz['PrognMartwy']
df_kalendarz['CzyMartwyWTymDniu'] = df_kalendarz['CzyMartwyWTymDniu'].astype('boolean')
df_kalendarz.loc[df_kalendarz['DataOstatniejSprzedazy'].isna(), 'CzyMartwyWTymDniu'] = pd.NA

print(df_kalendarz['CzyMartwyWTymDniu'].value_counts(dropna=False))
print(f"\nPróg indywidualny — opis:\n{df_kalendarz['PrognMartwy'].describe()}")

CzyMartwyWTymDniu
False    8786996
True       15519
Name: count, dtype: Int64

Próg indywidualny — opis:
count    8.792773e+06
mean     5.533320e+02
std      3.713226e+01
min      5.500000e+02
25%      5.500000e+02
50%      5.500000e+02
75%      5.500000e+02
max      1.420500e+03
Name: PrognMartwy, dtype: float64


In [ ]:

# ============================================================
# CzySezonowy — informacyjna flaga (niezależna od CzyMartwy)
# ============================================================

# Zobaczmy najpierw rozkład, żeby próg nie był z sufitu
print(df_kalendarz.drop_duplicates('TowId')['PrzerwaHistoryczna95pct'].describe())
print(df_kalendarz.drop_duplicates('TowId')['PrzerwaHistoryczna95pct'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

count    11846.000000
mean        44.195003
std         80.849101
min          1.000000
25%          7.950000
50%         16.000000
75%         39.937500
max        947.000000
Name: PrzerwaHistoryczna95pct, dtype: float64
0.50     16.0000
0.75     39.9375
0.90    109.8000
0.95    199.0000
0.99    400.6025
Name: PrzerwaHistoryczna95pct, dtype: float64


In [ ]:
PROG_SEZONOWY = 100  # dni — zgodny z wcześniejszym prog_zagrozony, tuż przy 90. percentylu (109.8)

towid_sezonowe = (
    df_kalendarz.drop_duplicates('TowId')
    .loc[lambda d: d['PrzerwaHistoryczna95pct'] >= PROG_SEZONOWY, 'TowId']
)

df_kalendarz['CzySezonowy'] = df_kalendarz['TowId'].isin(towid_sezonowe)

print(df_kalendarz.drop_duplicates('TowId')['CzySezonowy'].value_counts())

CzySezonowy
False    11184
True      1297
Name: count, dtype: int64


In [ ]:
df_kalendarz['CzySezonowy'] = df_kalendarz['CzySezonowy'].astype('int8')  
df_kalendarz['CzyMartwyWTymDniu'] = df_kalendarz['CzyMartwyWTymDniu'].astype('int8') 
df_kalendarz.sample()

,TowId,Data,DokId,Kolejnosc,NrPozycji,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,...,NazwaTowCleanName,NazwaAsortCleanName,JestMartwyHistorycznie,JestWazony,DataOstatniejSprzedazy,DniOdSprzedazy,PrzerwaHistoryczna95pct,PrognMartwy,CzyMartwyWTymDniu,CzySezonowy
1061618,5086,2023-09-14,1394985,1.0,1.0,4.0,1.0,0.0,4.99,4.99,...,lody grzeski rozek 110ml colian,lody,1,False,2023-09-14,0,17.25,550.0,0,0


In [ ]:
df_kalendarz.to_parquet(
    "../../dane/interim/fact_inka_hard_flagged_wazone.parquet",
    compression='zstd',
    index=False
)
print(f"Zapisano: {df_kalendarz.shape}")

Zapisano: (8802515, 40)


In [ ]:
# suma kontrolna
nazwa_pliku = "fact_inka_hard_flagged_wazone.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")
#Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   7687cf50ac8d617b1975ae3f76bf4401560fefabc47d9ddbaf72f908c10e2612

Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   7687cf50ac8d617b1975ae3f76bf4401560fefabc47d9ddbaf72f908c10e2612
